# 28 - DE sink: perturbation-level consistency

Checks whether the accDM run satisfies the Einstein trace equation (Ma & Bertschinger 21c), which
CLASS evaluates but never uses to evolve $h'$. In synchronous gauge, with CLASS density units,

$$h'' + 2\mathcal{H}h' - 2k^2\eta + 9a^2\delta p_{\rm tot} \;=\; \frac{3a^2}{\mathcal{H}}\sum_i \delta Q_i ,$$

and the right side vanishes for any model that conserves energy. For accDM the parent loses
$a\Gamma\rho_p\delta_p$ and the daughter gains $(1+\eta)\,a\Gamma\rho_p\delta_p$, so
$\sum_i\delta Q_i = \eta\,a\Gamma\rho_p\delta_p$.

**Method.** Integrate the trace equation for $h'_B$, driven by the code's own $h'_A$ (set from the
energy constraint), and report the drift $\Delta_h = (h'_B - h'_A)/\max|h'_A|$. The prediction is the
running integral of the right side.

**Expectation.** The DE sink (`acc_de_sink = yes`, option A) is homogeneous: $\delta\rho_{\rm DE} = 0$.
It fixes the background but not $\sum\delta Q$, so $\Delta_h$ should be the same with the sink on
and off. Its size decides whether option B (interacting-vacuum perturbations) is needed.

The daughter uses the `qm_acc_birth` grid (`ncdm_quadrature_strategy = 5`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline
from classy import Class

BASE = {'omega_b': 0.022383, 'H0': 67.32, 'A_s': 2.1005829616811546e-9,
        'n_s': 0.96605, 'tau_reio': 0.0543, 'N_ur': 0.00441}
OMEGA_CDM0 = 0.12011
A_REC = 1.0/1091.0
K_OUT = [0.01, 0.1, 1.0]                    # 1/Mpc
A_START = 1e-3
FIDUCIAL = dict(f_acc=0.1, eta=0.1, kappa=12.1, a_t=0.133)
N_Q = 51                                    # qm_acc_birth daughter bins (odd)

# rk evolver (ndf15 cannot handle the daughter hierarchy); exact ncdm hierarchy (no fluid)
COMMON = {'output': 'mPk', 'gauge': 'synchronous', 'evolver': 0,
          'k_output_values': ','.join(str(k) for k in K_OUT),
          'P_k_max_1/Mpc': 2.0, 'z_max_pk': 3.0, 'ncdm_fluid_approximation': 3}


def lcdm_params():
    return {**BASE, **COMMON, 'omega_cdm': OMEGA_CDM0,
            'N_ncdm': 1, 'deg_ncdm': '3', 'm_ncdm': '0.02', 'T_ncdm': '0.71611',
            'ncdm_quadrature_strategy': '0', 'ncdm_N_momentum_bins': '15'}


def accdm_params(f_acc, eta, kappa, a_t, sink, mass=1e16):
    # omega_cdm rescaled so the total matter at recombination matches LCDM
    ocdm = OMEGA_CDM0/(1 + f_acc*(1 - A_REC**kappa)/(1 + (A_REC/a_t)**kappa))
    return {**BASE, **COMMON, 'omega_cdm': ocdm,
            'vary_Gamma_acc': 'yes', 'kappa_acc': kappa, 'a_t_acc': a_t,
            'f_acc': f_acc, 'eta_acc': eta, 'm_acc_in_GeV': mass, 'm_cdm_in_GeV': mass,
            'N_ncdm': 2, 'deg_ncdm': '3, 1', 'm_ncdm': '0.02, {:.6e}'.format(mass*1e9),
            'T_ncdm': '0.71611, 1', 'ncdm_quadrature_strategy': '0, 5',
            'ncdm_N_momentum_bins': '15, {:d}'.format(N_Q),
            'acc_de_sink': 'yes' if sink else 'no'}


def run(params):
    cosmo = Class()
    cosmo.set(params)
    try:
        cosmo.compute()
        perts = cosmo.get_perturbations()['scalar']
        bg = cosmo.get_background()
    finally:
        cosmo.struct_cleanup()
        cosmo.empty()
    return perts, bg


def bg_spline(bg, key):
    """Cubic spline of a background column in ln a (in log for positive columns)."""
    a = 1.0/(1.0 + np.asarray(bg['z']))
    order = np.argsort(a)
    x, y = np.log(a[order]), np.asarray(bg[key])[order]
    keep = np.append(np.diff(x) > 0, True)
    x, y = x[keep], y[keep]
    if np.all(y > 0):
        s = CubicSpline(x, np.log(y))
        return lambda aa: np.exp(s(np.log(aa)))
    s = CubicSpline(x, y)
    return lambda aa: s(np.log(aa))


def cum_trapz(y, x):
    """Cumulative trapezoid; robust to the jumps at approximation switches."""
    return np.concatenate(([0.0], np.cumsum(0.5*(y[1:] + y[:-1])*np.diff(x))))


def trace_drift(pk, bg, k, eta):
    """Measured drift Delta_h and its prediction from sum dQ = eta a Gamma rho_p delta_p."""
    tau = np.asarray(pk['tau [Mpc]'])
    keep = np.append(np.diff(tau) > 0, True)     # CLASS repeats tau at approximation switches
    a = np.asarray(pk['a'])[keep]
    sel = a >= A_START
    col = lambda key: np.asarray(pk[key])[keep][sel]
    tau, a = tau[keep][sel], a[sel]
    hp, eta_m, dp = col('h_prime'), col('eta'), col('delta_p_tot')

    Hcal = a*bg_spline(bg, 'H [1/Mpc]')(a)
    hp_B = hp[0] + cum_trapz(-2.0*Hcal*hp + 2.0*k*k*eta_m - 9.0*a**2*dp, tau)
    env = np.maximum.accumulate(np.abs(hp))
    out = dict(a=a, Delta_h=(hp_B - hp)/env, pred=np.zeros_like(a))
    if '(.)rho_acc_cdm' in bg:
        dQ = (eta*a*bg_spline(bg, 'Gamma_acc')(a)*bg_spline(bg, '(.)rho_acc_cdm')(a)
              *col('delta_dcdm'))
        out['pred'] = -cum_trapz(3.0*a**2*dQ/Hcal, tau)/env
    return out


def floor(d, a_t):
    """max |Delta_h| before injection (a < a_t/2): the run's own noise floor."""
    return float(np.max(np.abs(d['Delta_h'][d['a'] < 0.5*a_t])))


print('helpers ready: daughter bins = {}, k = {}'.format(N_Q, K_OUT))

## LCDM null

No source term, so $\Delta_h$ measures the estimator's own floor.

In [ ]:
perts_l, bg_l = run(lcdm_params())
missing = {'delta_p_tot', 'eta', 'h_prime'} - set(perts_l[0])
assert not missing, 'k_output columns missing: {}'.format(missing)

null = {k: trace_drift(pk, bg_l, k, 0.0) for k, pk in zip(K_OUT, perts_l)}
NULL = max(float(np.max(np.abs(d['Delta_h']))) for d in null.values())
for k in K_OUT:
    print('k = {:<5g} max|Delta_h| = {:.2e}   at a=1: {:+.2e}'.format(
        k, float(np.max(np.abs(null[k]['Delta_h']))), float(null[k]['Delta_h'][-1])))
print('LCDM null floor: {:.2e}'.format(NULL))

## accDM, sink off vs on

Same parameters, same grid; only `acc_de_sink` differs.

In [ ]:
drift = {}
for sink in (False, True):
    perts, bg = run(accdm_params(**FIDUCIAL, sink=sink))
    drift[sink] = {k: trace_drift(pk, bg, k, FIDUCIAL['eta']) for k, pk in zip(K_OUT, perts)}

print('accDM', FIDUCIAL)
print('{:>6} {:>13} {:>13} {:>13} {:>10} {:>10}'.format(
    'k', 'off: D_h(1)', 'on: D_h(1)', 'on: pred(1)', 'meas/pred', 'floor'))
for k in K_OUT:
    off, on = drift[False][k], drift[True][k]
    m, p = float(on['Delta_h'][-1]), float(on['pred'][-1])
    print('{:>6g} {:>+13.3e} {:>+13.3e} {:>+13.3e} {:>10.3f} {:>10.2e}'.format(
        k, float(off['Delta_h'][-1]), m, p, m/p, floor(on, FIDUCIAL['a_t'])))

fig, axes = plt.subplots(1, len(K_OUT), figsize=(12, 3.6), sharey=True, constrained_layout=True)
for ax, k in zip(axes, K_OUT):
    ax.semilogx(drift[False][k]['a'], drift[False][k]['Delta_h'], label='sink off')
    ax.semilogx(drift[True][k]['a'], drift[True][k]['Delta_h'], label='sink on')
    ax.semilogx(drift[True][k]['a'], drift[True][k]['pred'], 'k--', lw=1, label='prediction')
    ax.axvline(FIDUCIAL['a_t'], color='grey', ls=':')
    ax.set_title(r'$k = {:g}\ \mathrm{{Mpc}}^{{-1}}$'.format(k))
    ax.set_xlabel(r'$a$')
    ax.grid(alpha=0.3)
axes[0].set_ylabel(r'$\Delta_h$')
axes[0].legend()
plt.show()

## Verdict

In [ ]:
worst_on = max(abs(float(drift[True][k]['Delta_h'][-1])) for k in K_OUT)
worst_floor = max(floor(drift[True][k], FIDUCIAL['a_t']) for k in K_OUT)
change = max(abs(float(drift[True][k]['Delta_h'][-1]) - float(drift[False][k]['Delta_h'][-1]))
             /abs(float(drift[False][k]['Delta_h'][-1])) for k in K_OUT)

print('LCDM null floor                    : {:.2e}'.format(NULL))
print('largest |Delta_h(1)|, sink on      : {:.2e}  (own pre-injection floor {:.2e})'.format(
    worst_on, worst_floor))
print('largest relative change off -> on  : {:.1%}'.format(change))
print()
if worst_on > 3*worst_floor:
    print('Perturbation-level violation remains with the sink on (expected for option A).')
    print('|Delta_h| ~ {:.0e} of max|h\'| is the size option B would remove.'.format(worst_on))
else:
    print('No perturbation-level violation above the floor with the sink on.')